# What each filter responds to

Gradient ascent in input space: synthesise the image that maximally excites a chosen filter, and read the network's visual vocabulary directly.

**Runs on:** CPU — about 4 minutes (GPU: 1 minute) &nbsp;·&nbsp; **Slides:** [Chapter 10 — Interpreting What ConvNets Learn](../../../course-web-slides/ch10/index.html) &nbsp;·&nbsp; **Section:** 02 — Visualizing convnet filters

---

## A pretrained model, for a richer vocabulary

In [ ]:
import keras
import numpy as np
import matplotlib.pyplot as plt

model = keras.applications.xception.Xception(
    weights="imagenet", include_top=False)

for layer in model.layers:
    if isinstance(layer, (keras.layers.Conv2D, keras.layers.SeparableConv2D)):
        print(layer.name)

## A feature extractor for one layer

In [ ]:
layer_name = "block3_sepconv1"
layer = model.get_layer(name=layer_name)
feature_extractor = keras.Model(inputs=model.input, outputs=layer.output)

activation = feature_extractor(
    keras.applications.xception.preprocess_input(
        np.random.uniform(size=(1, 200, 200, 3)) * 255))
print("activation shape:", activation.shape)

## The loss: mean activation of one filter

In [ ]:
from keras import ops

def compute_loss(image, filter_index):
    activation = feature_extractor(image)
    # Avoid the border, where padding artifacts dominate.
    filter_activation = activation[:, 2:-2, 2:-2, filter_index]
    return ops.mean(filter_activation)

**We are not minimising a loss here — we are maximising an activation.** The optimizer's direction is reversed, and the thing being updated is the *image*, not the weights.

## Gradient ascent

In [ ]:
import tensorflow as tf

@tf.function
def gradient_ascent_step(image, filter_index, learning_rate):
    with tf.GradientTape() as tape:
        tape.watch(image)
        loss = compute_loss(image, filter_index)
    grads = tape.gradient(loss, image)
    grads = tf.math.l2_normalize(grads)       # normalize: makes lr predictable
    image += learning_rate * grads
    return image

img_width = img_height = 200

def generate_filter_pattern(filter_index, iterations=30, learning_rate=10.):
    image = tf.random.uniform(minval=0.4, maxval=0.6,
                              shape=(1, img_width, img_height, 3))
    for _ in range(iterations):
        image = gradient_ascent_step(image, filter_index, learning_rate)
    return image[0].numpy()

def deprocess_image(image):
    image -= image.mean()
    image /= image.std() + 1e-5
    image *= 64
    image += 128
    image = np.clip(image, 0, 255).astype("uint8")
    return image[25:-25, 25:-25, :]

plt.figure(figsize=(4, 4))
plt.imshow(deprocess_image(generate_filter_pattern(filter_index=2)))
plt.axis("off"); plt.title(f"{layer_name}, filter 2"); plt.show()

> **Note** — `l2_normalize` on the gradient is what makes a single learning rate work across layers of wildly different activation scales. Without it you would retune the step size for every layer.

## A grid of filters

In [ ]:
all_images = []
for filter_index in range(64):
    image = deprocess_image(generate_filter_pattern(filter_index))
    all_images.append(image)

margin, n = 5, 8
cropped = all_images[0].shape[0]
width = n * cropped + (n - 1) * margin
stitched = np.zeros((width, width, 3), dtype="uint8")
for i in range(n):
    for j in range(n):
        img = all_images[i * n + j]
        stitched[(cropped + margin) * i: (cropped + margin) * i + cropped,
                 (cropped + margin) * j: (cropped + margin) * j + cropped, :] = img

plt.figure(figsize=(11, 11))
plt.imshow(stitched); plt.axis("off")
plt.title(f"64 filters from {layer_name}")
plt.show()

## The same, at three depths

In [ ]:
for name in ["block2_sepconv1", "block4_sepconv1", "block10_sepconv1"]:
    layer = model.get_layer(name=name)
    feature_extractor = keras.Model(inputs=model.input, outputs=layer.output)
    imgs = [deprocess_image(generate_filter_pattern(i)) for i in range(8)]
    fig, axes = plt.subplots(1, 8, figsize=(14, 2))
    for ax, im in zip(axes, imgs):
        ax.imshow(im); ax.axis("off")
    fig.suptitle(name, y=1.05)
    plt.show()

A clear progression:

- **Early** — simple edges and colours, close to Gabor filters.
- **Middle** — textures: feathers, eyes, foliage, grids.
- **Late** — recognisable object parts: whole feathers, dog faces, bird beaks.

**Nobody designed this hierarchy.** It falls out of backpropagation on labelled photographs, and it resembles the organisation of the primate visual cortex closely enough to be worth remarking on.

## Dead filters

In [ ]:
dead = []
for i in range(64):
    p = generate_filter_pattern(i, iterations=20)
    if p.std() < 1e-3:
        dead.append(i)
print(f"{len(dead)} of 64 filters produced a flat image: {dead}")

Filters that never activate on anything. They are pure overhead, and their existence is one of the arguments for the pruning and quantization techniques in chapter 18.

---

## What to take away

- Gradient **ascent** on the input synthesises what a filter is looking for.
- Normalize the gradient so one learning rate works at every depth.
- The hierarchy — edges, textures, object parts — is learned, not designed.
- Some filters are dead. Finding them is the first step toward pruning.